# ESMM on Ali-CCP — full data (Colab)

Multi-task ESMM-style training on the **full** Ali-CCP split. **OOM safety:** prep uses bounded batch sizes (`STREAM_PARSE_CHUNK_ROWS`, `VOCAB_SCAN_ROWS_PER_BATCH`, `NORM_STREAM_BATCH_ROWS`). Training reads **one Parquet row group at a time** plus `TRAIN_BATCH_SIZE` micro-batches; evaluation scans test Parquet in chunks of `EVAL_BATCH_SIZE`. Separate **training throughput** toggles (AMP, prefetch, etc.) are optional speed knobs—see Config. No full-split `pd.read_parquet` / `read_table` on this path.

**Data:** https://tianchi.aliyun.com/dataset/408 — place `sample_train.tar` / `sample_test.tar` under `DATA_DIR` (see Config).

**Four run switches** (Config cell): set any combination of `True`/`False`:

1. **`RUN_BASELINE`** — default **ESMM** architecture (`ESMMModel`), row-group training, 5 epochs.
2. **`RUN_SHARED_BOTTOM`** — shared trunk + two heads (`ESMM_SharedBottom`).
3. **`RUN_MMOE`** — MMoE variant (`ESMM_MMoE`).
4. **`RUN_PLE`** — PLE variant (`ESMM_PLE`).

All enabled runs share the same full-split parsed Parquet, filtered sparse vocabs, and normalized train/test Parquet under `PROCESSED_FULL_DIR`. Delete the matching result JSON in `ROUND_RESULTS_DIR` (e.g. `baseline_results.json`, `exp_shared_bottom_results.json`, …) to re-train that leg.

**Implementation:** `esmm_ali_ccp_impl.py` (this notebook only sets paths, prep, and toggles).

**Colab / scp:** Git sync cell skips `git fetch` / `reset` by default (`SKIP_GIT_REPO_SYNC`); set `FORCE_GIT_SYNC=1` to track GitHub `main`.


In [ ]:
import os
if os.path.ismount('/content/drive'):
    print('Drive already mounted.')
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        print('Skipping drive mount (not in Colab UI or drive unavailable).')

In [ ]:
import os
if not os.path.exists('/content/drive'):
    print('Warning: Drive not mounted, some paths may not work.')
WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os, subprocess, shutil
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'
branch_name = 'main'

# Default True: do NOT git fetch/reset — required for scp + papermill (see intro markdown).
# Colab UI: set False here, or export FORCE_GIT_SYNC=1 before Run All, to track GitHub main.
SKIP_GIT_REPO_SYNC = True
if os.environ.get('FORCE_GIT_SYNC', '').strip().lower() in ('1', 'true', 'yes'):
    SKIP_GIT_REPO_SYNC = False

if IN_COLAB:
    git_marker = os.path.join(repo_dir, '.git')
    if SKIP_GIT_REPO_SYNC:
        if os.path.isdir(repo_dir):
            os.chdir(repo_dir)
            print('[SKIP_GIT_REPO_SYNC] Skipping git fetch/reset — using Drive copy (scp/papermill safe).')
        else:
            print('[SKIP_GIT_REPO_SYNC] Repo dir missing; cloning once (no reset).')
            subprocess.run(['git', 'clone', repo_url], check=True)
            os.chdir(repo_dir)
            subprocess.run(['git', 'checkout', branch_name], check=False)
    elif os.path.isdir(repo_dir) and os.path.isdir(git_marker):
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)
        subprocess.run(['git', 'reset', '--hard', f'origin/{branch_name}'], check=False)
    else:
        if os.path.exists(repo_dir):
            shutil.rmtree(repo_dir)
        subprocess.run(['git', 'clone', repo_url], check=True)
        os.chdir(repo_dir)
        subprocess.run(['git', 'fetch', '--all'], check=True)
        subprocess.run(['git', 'checkout', branch_name], check=False)

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'torch', 'pandas', 'numpy', 'scikit-learn',
                'matplotlib', 'seaborn', 'requests', 'tqdm', 'joblib',
                'pyarrow', 'psutil'], check=True)

In [ ]:
import os

PROJECT_NAME = 'esmm_experiment'
DATA_DIR = '/content/drive/MyDrive/colab/data/ali_ccp'
PROCESSED_FULL_DIR = os.path.join(DATA_DIR, 'processed_esmm_full_parquet')
ROUND_RESULTS_DIR = os.path.join(DATA_DIR, 'esmm_round_training_cache')
for _d in (PROCESSED_FULL_DIR, ROUND_RESULTS_DIR):
    os.makedirs(_d, exist_ok=True)

PREPROCESSED_TRAIN = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_train.parquet')
PREPROCESSED_TEST = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_test.parquet')
PREPROCESSED_SPARSE_VOCAB_CACHE = os.path.join(PROCESSED_FULL_DIR, 'preprocessed_sparse_vocab.pkl')

# Per-experiment result JSON (delete a file to force re-training that leg).
BASELINE_RESULTS_JSON = os.path.join(ROUND_RESULTS_DIR, 'baseline_results.json')
EXP_SHARED_BOTTOM_RESULTS_JSON = os.path.join(ROUND_RESULTS_DIR, 'exp_shared_bottom_results.json')
EXP_MMOE_RESULTS_JSON = os.path.join(ROUND_RESULTS_DIR, 'exp_mmoe_results.json')
EXP_PLE_RESULTS_JSON = os.path.join(ROUND_RESULTS_DIR, 'exp_ple_results.json')

# Remove these cache JSON names before the run: 'baseline' | 'shared_bottom' | 'mmoe' | 'ple'
CLEAN_EXPERIMENT_JSON = []  # e.g. ['baseline', 'mmoe']

CLEAN_PREPROCESSED_PARQUET = False
CLEAN_PREPROCESSED_VOCAB_CACHE = False
FORCE_REBUILD_PREPROCESSED_VOCAB = False

def _drive_remove(path, desc):
    if os.path.isfile(path):
        os.remove(path)
        print(f'[cleanup] Removed {desc}: {path}')

_CLEAN_MAP = {
    'baseline': BASELINE_RESULTS_JSON,
    'shared_bottom': EXP_SHARED_BOTTOM_RESULTS_JSON,
    'mmoe': EXP_MMOE_RESULTS_JSON,
    'ple': EXP_PLE_RESULTS_JSON,
}
for _k in CLEAN_EXPERIMENT_JSON:
    if _k not in _CLEAN_MAP:
        raise ValueError(f'Unknown CLEAN_EXPERIMENT_JSON key: {_k!r}')
    _drive_remove(_CLEAN_MAP[_k], f'{_k}_results.json')

if CLEAN_PREPROCESSED_PARQUET:
    _drive_remove(PREPROCESSED_TRAIN, 'preprocessed_train.parquet')
    _drive_remove(PREPROCESSED_TEST, 'preprocessed_test.parquet')
if CLEAN_PREPROCESSED_VOCAB_CACHE:
    _drive_remove(PREPROCESSED_SPARSE_VOCAB_CACHE, 'preprocessed_sparse_vocab.pkl')

# --- What to run (full data only) ---
RUN_BASELINE = True
RUN_SHARED_BOTTOM = False
RUN_MMOE = False
RUN_PLE = False

_RUN_ANY_EXPERIMENT = RUN_BASELINE or RUN_SHARED_BOTTOM or RUN_MMOE or RUN_PLE

# --- Prep streaming (bounded I/O while building artifacts; tune down if Colab OOMs) ---
# STREAM_PARSE_CHUNK_ROWS: rows buffered before each flush when writing parsed_*_full Parquet from CSV.
# VOCAB_SCAN_ROWS_PER_BATCH: PyArrow batch size when scanning each sparse column on train Parquet for counts.
# NORM_STREAM_BATCH_ROWS: PyArrow batch size when reading parsed_* and writing preprocessed_* (log1p dense).
STREAM_PARSE_CHUNK_ROWS = 500_000
VOCAB_SCAN_ROWS_PER_BATCH = 200_000
NORM_STREAM_BATCH_ROWS = 500_000

# --- Training & evaluation I/O (model forward on GPU; data from disk in bounded chunks) ---
# Training (`train_esmm_parquet_rowgroups`): one Parquet row group loaded at a time from PREPROCESSED_TRAIN,
# then shuffled/sliced into TRAIN_BATCH_SIZE micro-batches for the optimizer.
TRAIN_BATCH_SIZE = 4096

# Evaluation (`evaluate_esmm_*_streaming_parquet`): scan PREPROCESSED_TEST with PyArrow iter_batches;
# EVAL_BATCH_SIZE is how many rows are decoded from disk per chunk (same "bound RAM from Parquet" idea as training).
EVAL_BATCH_SIZE = 500_000

# --- Training throughput optimizations (speed / GPU–I/O overlap; not the same as batch sizes above) ---
# Turn off for debugging, reproducibility stress-tests, or if you hit CUDA/Arrow edge cases.

# Mixed precision on CUDA: faster matmuls; BCE still uses float32 on detached probs inside the trainer (impl).
# PLE: train_esmm_parquet_rowgroups forces use_amp=False for ESMM_PLE regardless of this flag.
TRAIN_USE_AMP = True

# Decode/prepare the next Parquet row group on a worker thread while the GPU trains on the current one.
TRAIN_PREFETCH_ROW_GROUPS = True

# Shuffle inside each row group and slice TRAIN_BATCH_SIZE in Python (typical). False = DataLoader on the
# full row-group tensor (legacy path; can use more host RAM).
TRAIN_USE_MANUAL_BATCHES = True

# Prefer Arrow-native decode per row group before tensorize; falls back to pandas if a batch fails.
TRAIN_READ_ROW_GROUPS_AS_ARROW = False

# torch.compile(model) on CUDA when supported; warmup overhead and different failure modes vs eager.
TRAIN_USE_TORCH_COMPILE = False

RANDOM_STATE = 42
EMBED_DIM = 18

# Baseline training only (`_run_baseline` -> `train_esmm_parquet_rowgroups`). Optional early-stop caps; None = full 5 epochs.
BASELINE_MAX_WALL_SECONDS = None
BASELINE_MAX_OPTIMIZER_STEPS = None
BASELINE_MAX_BATCHES_PER_EPOCH = None
BASELINE_MAX_ROW_GROUPS_PER_EPOCH = None

SPARSE_COLS = ['101', '121', '122', '124', '125', '126', '127', '128', '129',
               '205', '206', '207', '210', '216', '508', '509', '702', '853',
               '301', '109_14', '110_14', '127_14', '150_14']
DENSE_COLS = ['109_14', '110_14', '127_14', '150_14', '508', '509', '702', '853']
DENSE_FEAT_COLS = ['D' + c for c in DENSE_COLS]

print('Config (full-data experiments):')
print(f'  RUN_BASELINE={RUN_BASELINE}  RUN_SHARED_BOTTOM={RUN_SHARED_BOTTOM}  RUN_MMOE={RUN_MMOE}  RUN_PLE={RUN_PLE}')
print(f'  EMBED_DIM={EMBED_DIM}, results under {ROUND_RESULTS_DIR}/')
print(
    f'  Prep Parquet batches: parse={STREAM_PARSE_CHUNK_ROWS:,} vocab_scan={VOCAB_SCAN_ROWS_PER_BATCH:,} '
    f'norm={NORM_STREAM_BATCH_ROWS:,}'
)
print(
    f'  Train/eval I/O batching: train_batch_size={TRAIN_BATCH_SIZE:,} eval_batch_size={EVAL_BATCH_SIZE:,}'
)
print(
    f'  Train throughput opts: amp={TRAIN_USE_AMP} prefetch_rg={TRAIN_PREFETCH_ROW_GROUPS} '
    f'manual_batches={TRAIN_USE_MANUAL_BATCHES} arrow_rg={TRAIN_READ_ROW_GROUPS_AS_ARROW} compile={TRAIN_USE_TORCH_COMPILE}'
)


In [ ]:
import os
import sys
import tarfile
from pathlib import Path

_IMPL_DIR = (Path.cwd() / "experiments" / "20260404_ali_cpp_esmm").resolve()
if _IMPL_DIR.is_dir() and str(_IMPL_DIR) not in sys.path:
    sys.path.insert(0, str(_IMPL_DIR))

from esmm_ali_ccp_impl import (
    COMMON_FEATURES_TRAIN,
    COMMON_FEATURES_TEST,
    SAMPLE_SKELETON_TRAIN,
    SAMPLE_SKELETON_TEST,
    SAMPLE_TRAIN_TAR,
    SAMPLE_TEST_TAR,
    TRAIN_CSV,
    TEST_CSV,
    find_file_recursive,
)

os.makedirs(DATA_DIR, exist_ok=True)

train_tar_path = os.path.join(DATA_DIR, SAMPLE_TRAIN_TAR)
test_tar_path = os.path.join(DATA_DIR, SAMPLE_TEST_TAR)

has_archives = os.path.isfile(train_tar_path) and os.path.isfile(test_tar_path)
needs_extract = has_archives and (
    not find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) or
    not find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN)
)
if needs_extract:
    for arc in [SAMPLE_TRAIN_TAR, SAMPLE_TEST_TAR]:
        path = os.path.join(DATA_DIR, arc)
        if os.path.isfile(path):
            print(f'Extracting {arc}...')
            with tarfile.open(path, 'r:*') as tf:
                tf.extractall(DATA_DIR)
            print('Done.')

has_raw = (
    find_file_recursive(DATA_DIR, SAMPLE_SKELETON_TRAIN) is not None and
    find_file_recursive(DATA_DIR, COMMON_FEATURES_TRAIN) is not None
)
has_splits = os.path.isfile(os.path.join(DATA_DIR, TRAIN_CSV)) and os.path.isfile(os.path.join(DATA_DIR, TEST_CSV))

if _RUN_ANY_EXPERIMENT and not (has_raw or has_splits):
    raise FileNotFoundError(
        f'No Ali-CCP raw files in {DATA_DIR}. Need tar extracts (skeleton + common_features) or {TRAIN_CSV}/{TEST_CSV}.'
    )

if _RUN_ANY_EXPERIMENT:
    print('Raw data layout OK for full-split streaming parse.')
else:
    print('No experiments enabled; skipped raw layout check.')


In [ ]:
# Training, models, and Parquet helpers: `esmm_ali_ccp_impl.py`
from esmm_ali_ccp_impl import *
import esmm_ali_ccp_impl as _impl

_impl.DEFAULT_STREAM_PARSE_CHUNK_ROWS = STREAM_PARSE_CHUNK_ROWS
_impl.DEFAULT_VOCAB_SCAN_ROWS_PER_BATCH = VOCAB_SCAN_ROWS_PER_BATCH
_impl.DEFAULT_NORM_STREAM_BATCH_ROWS = NORM_STREAM_BATCH_ROWS
# impl name kept for backward compat; equals Config EVAL_BATCH_SIZE (eval Parquet chunk rows).
_impl.DEFAULT_EVAL_TEST_BATCH_ROWS = EVAL_BATCH_SIZE


In [ ]:
import gc
import os
import time

import psutil
import torch

# Full split: chunked parse (if needed), batched vocab scan, batched normalize — paths only, no full DataFrame.
if not _RUN_ANY_EXPERIMENT:
    print('Full-data prep skipped (all experiment flags False).')
else:
    t0 = time.time()
    p_train, p_test = ensure_full_split_parquet_streaming(
        DATA_DIR, PROCESSED_FULL_DIR, SPARSE_COLS, DENSE_COLS, DENSE_FEAT_COLS)
    summarize_split(p_train, p_test, summary_batch_rows=STREAM_PARSE_CHUNK_ROWS)
    print(f'Full split Parquet paths ready in {time.time() - t0:.1f}s')

    t1 = time.time()
    print('Filtered sparse vocabs (cache or Parquet scan)...')
    # vocabs: per sparse column, raw string -> int id (0 reserved UNK in model). sparse_cardinalities: embedding
    # table size per column (same order as SPARSE_COLS); passed as field_cardinalities into train_esmm_parquet_rowgroups.
    vocabs, sparse_cardinalities = load_or_build_sparse_vocabs_filtered_parquet(
        p_train, SPARSE_COLS, min_count=5, cache_path=PREPROCESSED_SPARSE_VOCAB_CACHE,
        force_rebuild=FORCE_REBUILD_PREPROCESSED_VOCAB,
        vocab_scan_rows_per_batch=VOCAB_SCAN_ROWS_PER_BATCH)
    print(f'Vocab ready in {time.time() - t1:.1f}s')

    if not (os.path.isfile(PREPROCESSED_TRAIN) and os.path.isfile(PREPROCESSED_TEST)):
        print('Streaming dense log1p -> normalized Parquet...')
        t2 = time.time()
        stream_normalize_parquet(
            p_train, PREPROCESSED_TRAIN, SPARSE_COLS, DENSE_FEAT_COLS,
            norm_stream_batch_rows=NORM_STREAM_BATCH_ROWS)
        stream_normalize_parquet(
            p_test, PREPROCESSED_TEST, SPARSE_COLS, DENSE_FEAT_COLS,
            norm_stream_batch_rows=NORM_STREAM_BATCH_ROWS)
        print(f'Normalize done in {time.time() - t2:.1f}s')
    else:
        print(f'Reusing {PREPROCESSED_TRAIN} / {PREPROCESSED_TEST} (set CLEAN_PREPROCESSED_PARQUET=True to rebuild)')

    gc.collect()
    print(f'RAM after prep: {psutil.Process().memory_info().rss / 1024**3:.1f} GB')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
import json
import os
import time
import gc

import numpy as np
import torch

if not _RUN_ANY_EXPERIMENT:
    print('Train skipped: all RUN_* flags False in Config.')
else:
    _TRAIN_KW_MTL = dict(
        epochs=5,
        batch_size=TRAIN_BATCH_SIZE,
        lr=1e-3,
        seed=RANDOM_STATE,
        embed_dim=EMBED_DIM,
        max_wall_seconds=None,
        max_optimizer_steps=None,
        max_batches_per_epoch=None,
        max_row_groups_per_epoch=None,
        use_amp=TRAIN_USE_AMP,
        prefetch_row_groups=TRAIN_PREFETCH_ROW_GROUPS,
        use_manual_batches=TRAIN_USE_MANUAL_BATCHES,
        read_row_groups_as_arrow=TRAIN_READ_ROW_GROUPS_AS_ARROW,
        use_torch_compile=TRAIN_USE_TORCH_COMPILE,
    )


    def _jsonify(o):
        if isinstance(o, dict):
            return {k: _jsonify(v) for k, v in o.items()}
        if isinstance(o, (list, tuple)):
            return [_jsonify(v) for v in o]
        if isinstance(o, (np.floating, np.integer)):
            return o.item()
        return o

    def _fmt(x):
        if x is None or (isinstance(x, float) and (x != x)):
            return 'nan'
        return f'{float(x):.4f}'


    def _run_baseline():
        path = BASELINE_RESULTS_JSON
        if not RUN_BASELINE:
            print('BASELINE: skipped (RUN_BASELINE=False)')
            return
        if os.path.isfile(path):
            with open(path) as f:
                row = json.load(f)
            print('BASELINE: cached', row)
            return
        print('\n' + '=' * 60)
        print('BASELINE — default ESMM (full data, row groups, 5 epochs)')
        print('=' * 60)
        t0 = time.time()
        model, _, meta = train_esmm_parquet_rowgroups(
            PREPROCESSED_TRAIN, vocabs, sparse_cardinalities, SPARSE_COLS, DENSE_FEAT_COLS,
            epochs=5,
            batch_size=TRAIN_BATCH_SIZE,
            lr=1e-3,
            seed=RANDOM_STATE,
            embed_dim=EMBED_DIM,
            max_wall_seconds=BASELINE_MAX_WALL_SECONDS,
            max_optimizer_steps=BASELINE_MAX_OPTIMIZER_STEPS,
            max_batches_per_epoch=BASELINE_MAX_BATCHES_PER_EPOCH,
            max_row_groups_per_epoch=BASELINE_MAX_ROW_GROUPS_PER_EPOCH,
            use_amp=TRAIN_USE_AMP,
            prefetch_row_groups=TRAIN_PREFETCH_ROW_GROUPS,
            use_manual_batches=TRAIN_USE_MANUAL_BATCHES,
            read_row_groups_as_arrow=TRAIN_READ_ROW_GROUPS_AS_ARROW,
            use_torch_compile=TRAIN_USE_TORCH_COMPILE,
        )
        wall = time.time() - t0
        cvr_k, _ = evaluate_esmm_cvr_streaming_parquet(
            model, PREPROCESSED_TEST, vocabs, SPARSE_COLS, DENSE_FEAT_COLS,
            eval_batch_rows=EVAL_BATCH_SIZE)
        ctcvr_k, _ = evaluate_esmm_ctcvr_streaming_parquet(
            model, PREPROCESSED_TEST, vocabs, SPARSE_COLS, DENSE_FEAT_COLS,
            eval_batch_rows=EVAL_BATCH_SIZE)
        n_params = int(sum(p.numel() for p in model.parameters()))
        out = {
            'CVR_AUC': float(cvr_k),
            'CTCVR_AUC': float(ctcvr_k),
            'wall_clock_seconds': int(wall),
            'num_parameters': n_params,
            'train_samples_per_sec': float(meta['samples_per_sec']),
            'train_wall_seconds': float(meta['train_wall_seconds']),
            'early_stop_reason': meta['early_stop_reason'],
        }
        print(f"  CVR_AUC={cvr_k:.4f} CTCVR_AUC={ctcvr_k:.4f} wall={wall:.0f}s params={n_params:,}")
        with open(path, 'w') as f:
            json.dump(out, f)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    def _run_arch(name, run_flag, path, ctor, ctor_kw, desc):
        if not run_flag:
            print(f'{name}: skipped (flag False)')
            return
        if os.path.isfile(path):
            with open(path) as f:
                row = json.load(f)
            print(f'{name}: cached', {k: row.get(k) for k in ('CTCVR_AUC', 'CTR_AUC', 'CVR_AUC', 'num_parameters') if k in row})
            return
        print('\n' + '=' * 60)
        print(f'{name} — {desc}')
        print('=' * 60)
        t0 = time.time()
        model, _, meta = train_esmm_parquet_rowgroups(
            PREPROCESSED_TRAIN, vocabs, sparse_cardinalities, SPARSE_COLS, DENSE_FEAT_COLS,
            model_ctor=ctor,
            model_ctor_kwargs=ctor_kw if ctor_kw else None,
            **_TRAIN_KW_MTL,
        )
        wall = time.time() - t0
        n_params = int(sum(p.numel() for p in model.parameters()))
        metrics = evaluate_esmm_multitask_streaming_parquet(
            model, PREPROCESSED_TEST, vocabs, SPARSE_COLS, DENSE_FEAT_COLS,
            eval_batch_rows=EVAL_BATCH_SIZE)
        for mk, mv in metrics.items():
            print(f'  {mk}: {_fmt(mv)}')
        out = {
            **metrics,
            'wall_clock_seconds': int(wall),
            'num_parameters': n_params,
            'train_samples_per_sec': float(meta['samples_per_sec']),
            'train_wall_seconds': float(meta['train_wall_seconds']),
            'early_stop_reason': meta['early_stop_reason'],
        }
        with open(path, 'w') as f:
            json.dump(out, f)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    _run_baseline()
    _run_arch(
        'SHARED_BOTTOM', RUN_SHARED_BOTTOM, EXP_SHARED_BOTTOM_RESULTS_JSON,
        ESMM_SharedBottom, {'trunk_dims': (360, 200, 80)},
        'Shared trunk 360->200->80 + two heads',
    )
    _run_arch(
        'MMoE', RUN_MMOE, EXP_MMOE_RESULTS_JSON,
        ESMM_MMoE,
        {'num_experts': 4, 'expert_hidden': 360, 'd_model': 128, 'tower_hidden_ratio': 0.5},
        'MMoE E=4, expert_hidden=360, d_model=128',
    )
    _run_arch(
        'PLE', RUN_PLE, EXP_PLE_RESULTS_JSON,
        ESMM_PLE,
        {
            'd_model': 128,
            'expert_hidden': 256,
            'num_shared_experts': 1,
            'num_task_experts': 1,
            'dropout': 0.0,
        },
        '2-level PLE, 1 shared + 1 task expert per side',
    )

    print('\n' + '=' * 60)
    print('SUMMARY')
    print('=' * 60)
    for label, path in [
        ('BASELINE', BASELINE_RESULTS_JSON),
        ('SHARED_BOTTOM', EXP_SHARED_BOTTOM_RESULTS_JSON),
        ('MMoE', EXP_MMOE_RESULTS_JSON),
        ('PLE', EXP_PLE_RESULTS_JSON),
    ]:
        if not os.path.isfile(path):
            print(f'  {label}: (no cache)')
            continue
        with open(path) as f:
            r = json.load(f)
        if 'CTCVR_AUC' in r:
            print(f"  {label}: CTCVR_AUC={_fmt(r.get('CTCVR_AUC'))} CTR_AUC={_fmt(r.get('CTR_AUC'))} CVR_AUC={_fmt(r.get('CVR_AUC'))} params={r.get('num_parameters', 0):,}")
        else:
            print(f"  {label}: CVR_AUC={_fmt(r.get('CVR_AUC'))} CTCVR_AUC={_fmt(r.get('CTCVR_AUC'))} params={r.get('num_parameters', 0):,}")
